# Reference chunk materialization

Build the actual text chunks for the gated 512-content-token reference candidate. This notebook performs preprocessing with PySpark and preserves the earlier source snapshots.

## Bind the reviewed reference candidate

Use the corpus text handoff from Notebook 05 and the immutable offset and exclusion ledgers from Notebook 11. The identity includes the notebook sources and bound input manifests.

In [1]:
import hashlib
import json
import tempfile
from pathlib import Path

import nbformat
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pyspark.sql import functions as F

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/06_chunk_evaluation.ipynb").is_file()
)
notebook_path = ROOT / "notebooks/12_reference_chunk_materialization.ipynb"
n06_path = ROOT / "notebooks/06_chunk_evaluation.ipynb"
n06 = nbformat.read(n06_path, as_version=4)
source = {cell.id: cell.source for cell in n06.cells}
handoff = {"__name__": "__main__"}
for name in ("06-configure", "06-bind"):
    exec(compile(source[name], f"{n06_path.name}:{name}", "exec"), handoff)
spark = handoff["spark"]
parents = handoff["parents"]
sha256_file = handoff["sha256_file"]

qa_root = ROOT / ".cache/chunk-policy-review/program-chunk-qa-v1"
qa_manifest_path = qa_root / "MANIFEST.json"
qa_manifest = json.loads(qa_manifest_path.read_text(encoding="utf-8"))
audit = qa_manifest["reference_candidate_audit"]
if (
    audit["whole_parent_holds"] != 951
    or audit["remaining_held_parent_chunks"]
    or audit["remaining_exact_cross_split_groups"]
    or audit["invalid_candidate_chunks"]
    or audit["reference_content_token_budget"] != 512
):
    raise ValueError("The bound candidate does not have a clear reference offset gate.")

def bound_parquet(root, table, parts):
    paths = {
        path.relative_to(root).as_posix(): path
        for path in (root / table).rglob("*.parquet")
    }
    if not paths or set(paths) != {row["path"] for row in parts}:
        raise ValueError(f"The {table} file roster did not reconcile.")
    for part in parts:
        path = paths[part["path"]]
        if (
            sha256_file(path) != part["sha256"]
            or pq.ParquetFile(path).metadata.num_rows != part["rows"]
        ):
            raise ValueError(f"The {table} checksum or row count did not reconcile.")
    return spark.read.parquet(str(root / table))


offsets = bound_parquet(
    qa_root, "reference_candidate_offsets", qa_manifest["tables"]["reference_candidate_offsets"]
)
excluded_parents = bound_parquet(qa_root, "parent_holds", qa_manifest["tables"]["parent_holds"])
excluded_occurrences = bound_parquet(qa_root, "exact_holds", qa_manifest["tables"]["exact_holds"])
identity = {
    "source_handoff_manifest_sha256": sha256_file(handoff["handoff_manifest_path"]),
    "offset_candidate_manifest_sha256": sha256_file(qa_manifest_path),
    "materializer_cell_source_sha256": hashlib.sha256(
        json.dumps(
            [(cell.id, cell.source) for cell in nbformat.read(notebook_path, as_version=4).cells],
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest(),
}
output_base = ROOT / ".cache/chunk-materialization"
output_base.mkdir(mode=0o700, exist_ok=True)
output = output_base / "reference-v1"


Spark 4.2.1 · local[6] · Notebook 06


Verified 718,330 parents and 12 handoff files.
Notebook 05 handoff manifest SHA-256: ff507ad49c929a8352669e93e46151371a64b280857010559aecb38c639bbfcf


,ml_split,overlap_status,count
0,test,direct_evidence,381
1,test,no_cross_split_evidence_in_current_audit,35424
2,test,related_release_group,1
3,train,direct_evidence,1029
4,train,no_cross_split_evidence_in_current_audit,645383
5,train,related_release_group,2
6,validation,direct_evidence,382
7,validation,no_cross_split_evidence_in_current_audit,35722
8,validation,related_release_group,6


## Materialize text slices and verify stored Parquet

Slice by zero-based, half-open Unicode code point range. Keep original chunk ordinals where an occurrence was held. Verify the resulting text hash, lengths, reference-token counts, membership, and exact cross-split isolation before publication into a local versioned directory. Carry forward the upstream overlap status so lexical near-duplicate risks remain visible.

The text output contains corpus excerpts and stays local. Passing this materialization step confirms the integrity of the reference slices. Semantic QA has a separate receipt roster. A model-input release requires completed QA, a chosen tokenizer and its full input limit, and an exact cross-split replay whenever offsets change.

In [2]:
keys = ["release_row", "clean_sha256", "ml_split"]
occurrence_key = ["release_row", "start", "end", "chunk_sha256"]
if not output.exists():
    bodies = parents.select(
        *keys, "clean_text", "clean_chars", "native_identity", "release_group_id",
        "source_id", "source_host", "format", "publication_year", "publication_precision",
        "overlap_status", "cross_split_review_required",
    )
    derived = (
        offsets.join(bodies, keys)
        .withColumn(
            "chunk_text", F.substring(F.col("clean_text"), F.col("start") + 1, F.col("codepoints"))
        )
        .drop("clean_text")
    )
    with tempfile.TemporaryDirectory(prefix=".reference-v1-", dir=output_base) as temporary:
        stage = Path(temporary) / "snapshot"
        stage.mkdir(mode=0o700)
        chunks_dir = stage / "chunks"
        (
            derived.repartition(18, "release_row").write.mode("errorifexists")
            .option("compression", "zstd").partitionBy("ml_split").parquet(str(chunks_dir))
        )
        chunks = spark.read.parquet(str(chunks_dir))
        valid = (
            (F.col("start") >= 0)
            & (F.col("end") <= F.col("clean_chars"))
            & (F.col("end") - F.col("start") == F.col("codepoints"))
            & (F.length("chunk_text") == F.col("codepoints"))
            & (F.sha2("chunk_text", 256) == F.col("chunk_sha256"))
            & F.col("reference_token_count").between(1, audit["reference_content_token_budget"])
        )
        verification = chunks.agg(
            F.count("*").alias("chunks"),
            F.count_distinct("release_row").alias("parents"),
            F.count_distinct("release_row", "ordinal").alias("unique_chunk_keys"),
            F.count_if(~F.coalesce(valid, F.lit(False))).alias("invalid_chunks"),
        ).first().asDict()
        verification["held_parent_occurrences"] = chunks.join(
            F.broadcast(excluded_parents.select("release_row")), "release_row"
        ).count()
        verification["held_exact_occurrences"] = chunks.join(
            F.broadcast(excluded_occurrences.select(*occurrence_key)), occurrence_key
        ).count()
        verification["exact_cross_split_groups"] = (
            chunks.groupBy("chunk_sha256", "codepoints")
            .agg(F.min("ml_split").alias("first_split"), F.max("ml_split").alias("last_split"))
            .where(F.col("first_split") != F.col("last_split"))
            .count()
        )
        if (
            verification["chunks"] != audit["reference_candidate_chunks"]
            or verification["parents"] != audit["reference_candidate_parents"]
            or verification["unique_chunk_keys"] != verification["chunks"]
            or any(verification[name] for name in (
                "invalid_chunks", "held_parent_occurrences", "held_exact_occurrences",
                "exact_cross_split_groups",
            ))
        ):
            raise ValueError(f"The materialized reference slices failed verification: {verification}")

        splits = [
            row.asDict() for row in chunks.groupBy("ml_split").agg(
                F.count("*").alias("chunks"), F.count_distinct("release_row").alias("parents")
            ).orderBy("ml_split").collect()
        ]
        files = [
            {
                "path": path.relative_to(stage).as_posix(),
                "rows": pq.ParquetFile(path).metadata.num_rows,
                "sha256": sha256_file(path),
                "bytes": path.stat().st_size,
            }
            for path in sorted(chunks_dir.rglob("*.parquet"))
        ]
        result = {
            **identity,
            "artifact": "reference_chunk_materialization_v1",
            "selection_stage": "reference_candidate",
            "approved_for_model_input": False,
            "reference_encoding": audit["reference_encoding"],
            "reference_content_token_budget": audit["reference_content_token_budget"],
            "model_tokenizer_status": "pending_model_selection_and_validation",
            "semantic_qa_status_location": ".cache/chunk-policy-review/program-chunk-qa-v1/decisions",
            "verification": verification,
            "splits": splits,
            "files": files,
        }
        (stage / "MANIFEST.json").write_text(
            json.dumps(result, indent=2, sort_keys=True) + "\n", encoding="utf-8"
        )
        if output.exists():
            raise FileExistsError("Another writer created the reference snapshot.")
        stage.rename(output)

stored = json.loads((output / "MANIFEST.json").read_text(encoding="utf-8"))
if (
    any(stored.get(key) != value for key, value in identity.items())
    or stored["approved_for_model_input"]
    or sum(part["rows"] for part in stored["files"]) != audit["reference_candidate_chunks"]
):
    raise ValueError("The stored reference snapshot belongs to a different policy or roster.")
bound_parquet(output, "chunks", stored["files"])
display(pd.DataFrame(stored["splits"]))
print(
    f"Verified {stored['verification']['chunks']:,} reference text chunks in "
    f"{len(stored['files'])} local Parquet parts."
)
print(f"Snapshot: {output.relative_to(ROOT)}")
spark.stop()


,chunks,ml_split,parents
0,70021,test,35759
1,1370184,train,645539
2,71451,validation,36073


Verified 1,511,656 reference text chunks in 54 local Parquet parts.
Snapshot: .cache/chunk-materialization/reference-v1
